# Phase 3 Session 2 — NB1: Data Prep + Embedding Training

**Goal:** Train improved contrastive embedding with larger batch (128 in-batch negatives) and tau=0.12.

**Config:** `embedding_v2.yaml` (batch=128, tau=0.12, epochs=15, patience=5)

**Output:** Upload `/kaggle/working/outputs_p3s2_nb1/` as Kaggle dataset `p3s2-embedding` for nb2.

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import glob
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-absa-restaurant',
                  '/kaggle/input/semeval-2016-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
if KAGGLE_INPUT is None:
    inputs = glob.glob('/kaggle/input/**/semeval*train*.xml', recursive=True)
    assert inputs, 'Cannot find SemEval dataset'
    KAGGLE_INPUT = os.path.dirname(inputs[0])

print(f'Using: {KAGGLE_INPUT}')
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)
train_xml = glob.glob(f'{KAGGLE_INPUT}/*[Tt]rain*.xml')
test_xml = glob.glob(f'{KAGGLE_INPUT}/*[Tt]est*.xml') + glob.glob(f'{KAGGLE_INPUT}/*gold*')
shutil.copy(train_xml[0], 'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(test_xml[0], 'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')
print('XML files in place.')

## 1. Data Preparation

In [ ]:
!python scripts/01_prepare_data.py

## 2. Train Embedding

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
!python scripts/02_train_embedding.py --config configs/embedding_v2.yaml

In [ ]:
import json

if os.path.exists('logs/embedding_v2.jsonl'):
    print(f'{"Epoch":<8} {"Loss":<10} {"R@1":<8} {"R@3":<8} {"R@5":<8}')
    print('-' * 42)
    with open('logs/embedding_v2.jsonl') as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<10.4f} "
                  f"{rec.get('recall@1', 0):<8.4f} {rec.get('recall@3', 0):<8.4f} "
                  f"{rec.get('recall@5', 0):<8.4f}")

## 3. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p3s2_nb1'
os.makedirs(output_dir, exist_ok=True)

# Embedding checkpoint
if os.path.exists('checkpoints/embedding/best.pt'):
    shutil.copy('checkpoints/embedding/best.pt',
                os.path.join(output_dir, 'embedding_best.pt'))
    size_mb = os.path.getsize('checkpoints/embedding/best.pt') / 1e6
    print(f'embedding_best.pt: {size_mb:.1f} MB')

# Processed data (needed by nb2, nb3, nb4)
data_out = os.path.join(output_dir, 'processed')
if os.path.exists('data/processed'):
    shutil.copytree('data/processed', data_out, dirs_exist_ok=True)
    for f in os.listdir(data_out):
        print(f'processed/{f}')

# Logs
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(output_dir, 'logs'), dirs_exist_ok=True)

print(f'\nOutputs saved to {output_dir}')
print('Upload this as Kaggle dataset: p3s2-embedding')